In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
# References for DistilBERT and DistilRoBERTa:
# - DistilBERT documentation: https://huggingface.co/docs/transformers/en/model_doc/distilbert
# - DistilBERT fine-tuning example: https://medium.com/@ahmettsdmr1312/fine-tuning-distilbert-for-emotion-classification-84a4e038e90e
# - DistilRoBERTa doccumentation: https://huggingface.co/distilbert/distilroberta-base
# - DistilRoBERTa fine-tuning example: https://medium.com/analytics-vidhya/distil-roberta-for-hate-speech-classification-and-a-conceptual-review-about-transformers-c283bd8ff827

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/4neutraldatasets/test_moderate.csv
/kaggle/input/4neutraldatasets/train_strict.csv
/kaggle/input/4neutraldatasets/test_strict.csv
/kaggle/input/4neutraldatasets/train_moderate.csv
/kaggle/input/4neutraldatasets/train_equal.csv
/kaggle/input/4neutraldatasets/train_bit_negative.csv
/kaggle/input/4neutraldatasets/test_bit_negative.csv
/kaggle/input/4neutraldatasets/test_equal.csv


In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, RobertaTokenizer, RobertaForSequenceClassification,AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import re

torch.manual_seed(42)
np.random.seed(42)

2024-08-26 02:14:14.312440: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-26 02:14:14.312565: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-26 02:14:14.441105: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
def preprocess_text(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE).lower()

def prepare_balanced_data(df, sentiment_column):
    min_class_count = df[sentiment_column].value_counts().min()
    return pd.concat([df[df[sentiment_column] == sentiment].sample(min_class_count, random_state=42) 
                      for sentiment in df[sentiment_column].unique()], ignore_index=True)

In [4]:
def tokenize_function(examples, tokenizer, max_length) :
    return tokenizer(examples['comment'], padding='max_length', truncation=True, max_length=max_length)

In [5]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 
            'f1': f1, 
            'precision': precision,
            'recall': recall}

In [6]:
def print_distribution(df, name, sentiment_column):
    print(f"\nDistribution for {name}:")
    print(df[sentiment_column].value_counts())
    print(df[sentiment_column].value_counts(normalize=True))



In [7]:
def train_and_evaluate(model, train_dataset, eval_dataset, model_name, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
    )
    
    trainer.train()
    
    return trainer.evaluate()

In [8]:
neutral_pairs = [
    ('/kaggle/input/4neutraldatasets/train_bit_negative.csv', '/kaggle/input/4neutraldatasets/test_bit_negative.csv', 'sentiment_bit_negative'),
    ('/kaggle/input/4neutraldatasets/train_equal.csv', '/kaggle/input/4neutraldatasets/test_equal.csv', 'sentiment_equal'),
    ('/kaggle/input/4neutraldatasets/train_moderate.csv', '/kaggle/input/4neutraldatasets/test_moderate.csv', 'sentiment_moderate'),
    ('/kaggle/input/4neutraldatasets/train_strict.csv', '/kaggle/input/4neutraldatasets/test_strict.csv', 'sentiment_strict')
]

models = [
    ('distilbert', DistilBertTokenizer, DistilBertForSequenceClassification, 'distilbert-base-uncased'),
    ('distilroberta', AutoTokenizer, AutoModelForSequenceClassification, 'distilroberta-base')
]


In [9]:
results = {}

for train_file, test_file, sentiment_column in neutral_pairs:
    print(f"\nProcessing {sentiment_column} dataset")
    
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    
    for df in [train_df, test_df]:
        df['comment'] = df['comment'].apply(preprocess_text)
    
    print_distribution(train_df, "Original Train Set", sentiment_column)
    
    balanced_train_df = prepare_balanced_data(train_df, sentiment_column)
    balanced_test_df = prepare_balanced_data(test_df, sentiment_column)
    
    print_distribution(balanced_train_df, "Balanced Train Set", sentiment_column)
    print_distribution(balanced_test_df, "Balanced Test Set", sentiment_column)
    
    sentiment_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
    balanced_train_df['label'] = balanced_train_df[sentiment_column].map(sentiment_mapping)
    balanced_test_df['label'] = balanced_test_df[sentiment_column].map(sentiment_mapping)
    
    for model_name, tokenizer_class, model_class, model_path in models:
        print(f"\nTraining {model_name} on {sentiment_column} dataset")
        
        tokenizer = tokenizer_class.from_pretrained(model_path)
        model = model_class.from_pretrained(model_path, num_labels=3)
        
        train_dataset = Dataset.from_pandas(balanced_train_df).map(
            lambda x: tokenize_function(x, tokenizer, max_length=64), batched=True)
        eval_dataset = Dataset.from_pandas(balanced_test_df).map(
            lambda x: tokenize_function(x, tokenizer, max_length=64), batched=True)
        
        training_args = TrainingArguments(
            output_dir=f'./{model_name}_{sentiment_column}_results',
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            warmup_steps=500,
            weight_decay=0.01,
            evaluation_strategy="epoch",
            save_strategy="no",
            load_best_model_at_end=False,
            metric_for_best_model="accuracy",
            report_to="none"
        )
        
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            compute_metrics=compute_metrics,
        )
        
        trainer.train()
        eval_results = trainer.evaluate()
        
        results[f"{model_name}_{sentiment_column}"] = eval_results
        
        print(f"\nResults for {model_name} on {sentiment_column}:")
        for metric, score in eval_results.items():
            print(f"{metric}: {score:.4f}")
        


Processing sentiment_bit_negative dataset

Distribution for Original Train Set:
sentiment_bit_negative
Positive    1148684
Neutral       96843
Negative      86702
Name: count, dtype: int64
sentiment_bit_negative
Positive    0.862227
Neutral     0.072692
Negative    0.065080
Name: proportion, dtype: float64

Distribution for Balanced Train Set:
sentiment_bit_negative
Positive    86702
Negative    86702
Neutral     86702
Name: count, dtype: int64
sentiment_bit_negative
Positive    0.333333
Negative    0.333333
Neutral     0.333333
Name: proportion, dtype: float64

Distribution for Balanced Test Set:
sentiment_bit_negative
Positive    21378
Neutral     21378
Negative    21378
Name: count, dtype: int64
sentiment_bit_negative
Positive    0.333333
Neutral     0.333333
Negative    0.333333
Name: proportion, dtype: float64

Training distilbert on sentiment_bit_negative dataset


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

... (58 outputs left out)

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.588600,0.578649,0.743764,0.744130,0.750617,0.743764
2,0.543800,0.581016,0.747928,0.744970,0.743820,0.747928
3,0.494200,0.591767,0.749859,0.749400,0.749282,0.749859


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '



Results for distilroberta on sentiment_strict:
eval_loss: 0.5918
eval_accuracy: 0.7499
eval_f1: 0.7494
eval_precision: 0.7493
eval_recall: 0.7499
eval_runtime: 70.3996
eval_samples_per_second: 757.4190
eval_steps_per_second: 11.8470
epoch: 3.0000


In [10]:
print("\nOverall Results:")
for key, value in results.items():
    print(f"\nResults for {key}:")
    for metric, score in value.items():
        print(f"{metric}: {score:.4f}")

best_result = max(results, key=lambda x: results[x]['eval_accuracy'])
print(f"\nBest model and dataset: {best_result}")
print("Scores:")
for metric, score in results[best_result].items():
    print(f"{metric}: {score:.4f}")


Overall Results:

Results for distilbert_sentiment_bit_negative:
eval_loss: 0.5849
eval_accuracy: 0.7598
eval_f1: 0.7597
eval_precision: 0.7596
eval_recall: 0.7598
eval_runtime: 81.5963
eval_samples_per_second: 785.9920
eval_steps_per_second: 12.2920
epoch: 3.0000

Results for distilroberta_sentiment_bit_negative:
eval_loss: 0.5537
eval_accuracy: 0.7685
eval_f1: 0.7676
eval_precision: 0.7668
eval_recall: 0.7685
eval_runtime: 84.6388
eval_samples_per_second: 757.7380
eval_steps_per_second: 11.8500
epoch: 3.0000

Results for distilbert_sentiment_equal:
eval_loss: 0.5870
eval_accuracy: 0.7647
eval_f1: 0.7628
eval_precision: 0.7616
eval_recall: 0.7647
eval_runtime: 81.6278
eval_samples_per_second: 785.6890
eval_steps_per_second: 12.2870
epoch: 3.0000

Results for distilroberta_sentiment_equal:
eval_loss: 0.5482
eval_accuracy: 0.7726
eval_f1: 0.7696
eval_precision: 0.7685
eval_recall: 0.7726
eval_runtime: 85.3425
eval_samples_per_second: 751.4890
eval_steps_per_second: 11.7530
epoch: 3.000